# 🔥 Advanced · QLoRA — fine-tune an LLM

> 🔥 **Advanced / heavy lab.** Fine-tune a pretrained instruct LLM on your data, cheaply, with 4-bit QLoRA.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **20–40 min** including downloads. Built on the official **[TRL + PEFT + bitsandbytes](https://github.com/huggingface/trl)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

The practical counterpart to the from-scratch **nanoGPT** lab: instead of pretraining, you adapt a pretrained model by training tiny LoRA adapters on top of a 4-bit-quantized base — fits on a free T4.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

## 1 · Load a small instruct model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
base = "Qwen/Qwen2.5-0.5B-Instruct"   # swap for Llama-3.2-1B-Instruct, etc.
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")

## 2 · Data — a small instruction dataset
Replace with your own `{"text": "...chat-formatted..."}` records to teach your task.

In [ ]:
from datasets import load_dataset
ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train")   # ~1k chat samples, 'text' field
print(ds, "\n---\n", ds[0]["text"][:300])

## 3 · Attach LoRA adapters and train (TRL SFTTrainer)

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM", target_modules="all-linear")
args = SFTConfig(output_dir="qlora-out", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                 max_steps=200, learning_rate=2e-4, logging_steps=20, bf16=True,
                 max_seq_length=512, dataset_text_field="text", report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=peft_cfg, processing_class=tok)
trainer.train()

## 4 · Try it, then save the adapter

In [ ]:
def chat(msg, n=200):
    ids = tok.apply_chat_template([{"role": "user", "content": msg}], add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=n, do_sample=True, temperature=0.7)
    print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
chat("Explain LoRA fine-tuning in one paragraph.")
trainer.save_model("qlora-adapter")   # tiny — just the adapters

## Troubleshooting & next steps
- **TRL API drift**: newer TRL uses `SFTConfig`/`processing_class`; older uses `tokenizer=` and `dataset_text_field` on the trainer. Match your installed version (`pip show trl`).
- Merge adapters for deployment: `model.merge_and_unload()`.
- Then **align** the model with preferences → the DPO lab. For a much faster path, try **Unsloth**.